# ❤️ ECG Anomaly Detection — Phase 3: Model Training

This notebook trains the hybrid **1D-CNN + Transformer** model on the [PTB-XL dataset](https://physionet.org/content/ptb-xl/1.0.3/) (21,799 12-lead ECGs at 500 Hz) to detect 10 cardiac conditions with calibrated confidence scores.

## Before you start
1. **Runtime → Change runtime type → T4 GPU** (or A100 if you have Colab Pro)
2. **Runtime → Run all** — cells will execute sequentially
3. Allow ~20-30 min for the PTB-XL download (~3 GB), then ~2-4 hours for training

Checkpoints are saved to Google Drive so training can be resumed if the session disconnects.

---
| Condition | PTB-XL SCP codes |
|-----------|------------------|
| Normal Sinus Rhythm | NORM |
| Atrial Fibrillation | AFIB, AFLT |
| ST Elevation (STEMI) | AMI, IMI, STE\_ ... |
| Left Bundle Branch Block | LBBB, CLBBB |
| Right Bundle Branch Block | RBBB, CRBBB, IRBBB |
| Left Ventricular Hypertrophy | LVH |
| Bradycardia | SBRAD, BRAD |
| Tachycardia | STACH, SVTACH |
| First Degree AV Block | 1AVB |
| Premature Ventricular Contraction | PVC, VPCS |

## 1. Mount Google Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Edit these paths if you want to store data/checkpoints elsewhere ──────
BASE_DIR       = '/content/drive/MyDrive/ecg_project'
DATA_DIR       = f'{BASE_DIR}/ptbxl'
CHECKPOINT_DIR = f'{BASE_DIR}/checkpoints'
LOG_DIR        = f'{BASE_DIR}/logs'
EVAL_DIR       = f'{BASE_DIR}/eval_output'
# ─────────────────────────────────────────────────────────────────────────

for d in [BASE_DIR, DATA_DIR, CHECKPOINT_DIR, LOG_DIR, EVAL_DIR]:
    os.makedirs(d, exist_ok=True)

print('Paths ready:')
for label, path in [('data', DATA_DIR), ('checkpoints', CHECKPOINT_DIR),
                     ('logs', LOG_DIR), ('eval', EVAL_DIR)]:
    print(f'  {label:15s}: {path}')

## 2. Install dependencies

In [ ]:
# wfdb is the only package Colab doesn't have pre-installed.
!pip install -q wfdb

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Clone / upload the Phase 3 code

In [ ]:
import os, sys, zipfile

os.makedirs('/content/phase3/ecg_model', exist_ok=True)
sys.path.insert(0, '/content/phase3')

# Option A (recommended): unzip a Drive archive you saved previously
zip_path = f'{BASE_DIR}/phase3.zip'
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    print('✓ Restored from Drive zip')

# Option B: clone from GitHub
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/phase3

# Option C (default): write modules inline below — run subsequent cells
else:
    print('No Drive zip found — writing code inline from this notebook.')
    print('TIP: after training, save a zip for faster recovery next session:')
    print('  import shutil')
    print('  shutil.make_archive(f"{BASE_DIR}/phase3", "zip", "/content/phase3")')


In [ ]:
%%writefile /content/phase3/ecg_model/__init__.py
"""ecg_model — Phase 3 ML training package."""
from .model import ECGAnomalyDetector, ModelConfig, CONDITION_NAMES, N_CONDITIONS, build_model, load_checkpoint
from .dataset import PTBXLDataset, load_ptbxl, make_dataloaders, download_ptbxl
from .train import TrainConfig, train
from .evaluate import evaluate, plot_training_history

In [ ]:
%%writefile /content/phase3/ecg_model/model.py
from __future__ import annotations
import math
from dataclasses import dataclass
import torch
import torch.nn as nn

CONDITION_NAMES = (
    "Normal Sinus Rhythm",
    "Atrial Fibrillation",
    "ST Elevation",
    "Left Bundle Branch Block",
    "Right Bundle Branch Block",
    "Left Ventricular Hypertrophy",
    "Bradycardia",
    "Tachycardia",
    "First Degree AV Block",
    "Premature Ventricular Contraction",
)
N_CONDITIONS = len(CONDITION_NAMES)

@dataclass
class ModelConfig:
    n_leads: int = 12
    n_samples: int = 5000
    n_conditions: int = N_CONDITIONS
    cnn_channels: tuple = (32, 64, 128, 128)
    cnn_kernel_sizes: tuple = (7, 5, 5, 3)
    cnn_pool_sizes: tuple = (2, 2, 2, 2)
    cnn_dropout: float = 0.1
    d_model: int = 128
    n_heads: int = 4
    n_transformer_layers: int = 2
    dim_feedforward: int = 512
    transformer_dropout: float = 0.1
    head_hidden: int = 256
    head_dropout: float = 0.3

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel, pool, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=kernel//2, bias=False),
            nn.BatchNorm1d(out_ch), nn.ReLU(inplace=True),
            nn.MaxPool1d(pool), nn.Dropout(dropout),
        )
    def forward(self, x): return self.block(x)

class LeadEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        chs = [1] + list(cfg.cnn_channels)
        self.blocks = nn.ModuleList([
            ConvBlock(chs[i], chs[i+1], cfg.cnn_kernel_sizes[i],
                      cfg.cnn_pool_sizes[i], cfg.cnn_dropout)
            for i in range(len(cfg.cnn_channels))
        ])
    def forward(self, x):
        B, L, N = x.shape
        x = x.reshape(B*L, 1, N)
        for blk in self.blocks: x = blk(x)
        _, C, T = x.shape
        return x.reshape(B, L, C, T)

class LearnablePositionalEncoding(nn.Module):
    def __init__(self, n_leads, max_frames, d_model):
        super().__init__()
        self.lead_emb = nn.Embedding(n_leads, d_model)
        self.time_emb = nn.Embedding(max_frames, d_model)
    def forward(self, x, n_leads, n_frames):
        dev = x.device
        leads = torch.arange(n_leads, device=dev).repeat_interleave(n_frames)
        times = torch.arange(n_frames, device=dev).repeat(n_leads)
        return x + self.lead_emb(leads) + self.time_emb(times)

class ECGAnomalyDetector(nn.Module):
    def __init__(self, cfg=None):
        super().__init__()
        if cfg is None: cfg = ModelConfig()
        self.cfg = cfg
        self.lead_encoder = LeadEncoder(cfg)
        t_prime = cfg.n_samples
        for p in cfg.cnn_pool_sizes: t_prime = t_prime // p
        self.t_prime = t_prime
        cnn_out_ch = cfg.cnn_channels[-1]
        self.input_proj = nn.Identity() if cnn_out_ch == cfg.d_model else nn.Linear(cnn_out_ch, cfg.d_model)
        self.pos_enc = LearnablePositionalEncoding(cfg.n_leads, self.t_prime + 32, cfg.d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.n_heads,
            dim_feedforward=cfg.dim_feedforward,
            dropout=cfg.transformer_dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_transformer_layers,
                                                  enable_nested_tensor=False)
        self.classifier = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.head_hidden),
            nn.LayerNorm(cfg.head_hidden), nn.ReLU(inplace=True),
            nn.Dropout(cfg.head_dropout),
            nn.Linear(cfg.head_hidden, cfg.n_conditions),
        )
        self.register_buffer("temperature", torch.ones(1))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        B, L, N = x.shape
        feats = self.lead_encoder(x)
        _, _, C, T = feats.shape
        feats = feats.permute(0,1,3,2).reshape(B, L*T, C)
        feats = self.input_proj(feats)
        feats = self.pos_enc(feats, n_leads=L, n_frames=T)
        feats = self.transformer(feats)
        return self.classifier(feats.mean(dim=1))

    @torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.sigmoid(self.forward(x) / self.temperature.clamp(min=0.05))

    def parameter_count(self):
        return {"total": sum(p.numel() for p in self.parameters()),
                "trainable": sum(p.numel() for p in self.parameters() if p.requires_grad)}

class TemperatureScaler:
    def __init__(self, model): self.model = model
    def fit(self, logits, labels, lr=0.01, max_iter=50):
        temperature = nn.Parameter(self.model.temperature.clone().requires_grad_(True))
        opt = torch.optim.LBFGS([temperature], lr=lr, max_iter=max_iter)
        crit = nn.BCEWithLogitsLoss()
        def step():
            opt.zero_grad()
            loss = crit(logits / temperature.clamp(min=0.05), labels)
            loss.backward()
            return loss
        opt.step(step)
        self.model.temperature.copy_(temperature.detach().clamp(min=0.05))
        return crit(logits / self.model.temperature, labels).item()

def build_model(cfg=None, device=None):
    m = ECGAnomalyDetector(cfg)
    if device: m = m.to(device)
    return m

def load_checkpoint(path, device="cpu", cfg=None):
    from dataclasses import fields
    ckpt = torch.load(path, map_location=device)
    cfg = cfg or ModelConfig(**{k: v for k, v in ckpt.get("cfg", {}).items()
                                if k in {f.name for f in fields(ModelConfig)}})
    model = ECGAnomalyDetector(cfg).to(device)
    model.load_state_dict(ckpt["model_state"])
    return model


## 4a. Set up Kaggle credentials

Upload your `kaggle.json` API token (from https://www.kaggle.com/settings → *Create New Token*).
This is required to download PTB-XL via the Kaggle dataset mirror.

In [ ]:
from google.colab import files
import json, os

print("Upload your kaggle.json (Kaggle → Settings → Create New API Token)")
uploaded = files.upload()          # select kaggle.json from your machine

token = json.loads(list(uploaded.values())[0])
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(token, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured ✓')


## 4b. Download PTB-XL via Kaggle (~3 GB)

Uses the Kaggle API mirror (`khyeh0719/ptb-xl-dataset`) — much more reliable than PhysioNet wget.
Skips automatically if the dataset is already present in Drive.

In [ ]:
import os, shutil, zipfile
from pathlib import Path

DATA_DIR = '/content/ptbxl'
TMP_DIR = '/content/ptbxl_tmp'
ZIP_PATH = '/content/ptb-xl-dataset.zip'

meta_path = Path(DATA_DIR) / 'ptbxl_database.csv'

if meta_path.exists():
    print(f'PTB-XL already present at {DATA_DIR} — skipping download.')

else:
    print('Downloading PTB-XL (~3GB)...')
    !pip install -q kaggle
    !kaggle datasets download -d khyeh0719/ptb-xl-dataset -p /content --force

    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(TMP_DIR)

    # 🔥 Directly point to known folder (no rglob)
    extracted_dir = Path(TMP_DIR) / 'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1'

    if not extracted_dir.exists():
        raise FileNotFoundError("Expected dataset folder not found after extraction.")

    print(f'Moving files to {DATA_DIR} ...')
    os.makedirs(DATA_DIR, exist_ok=True)

    for item in extracted_dir.iterdir():
        dest = Path(DATA_DIR) / item.name
        if not dest.exists():
            shutil.move(str(item), str(dest))

    # Cleanup
    shutil.rmtree(TMP_DIR, ignore_errors=True)
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)

    print('✅ Dataset ready!')

# Verify
import pandas as pd

if meta_path.exists():
    df = pd.read_csv(meta_path)
    print(f'\nPTB-XL loaded: {len(df):,} records')
    print(df[['patient_id', 'age', 'sex']].head())
else:
    print('❌ Still not found — something is off.')

## 5. Explore the dataset & class distribution

In [ ]:
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/phase3/')

from ecg_model.dataset import scp_to_labels, load_ptbxl, PTBXLDataset
from ecg_model.model import CONDITION_NAMES

train_ds, val_ds, test_ds = load_ptbxl(DATA_DIR)
print(f'Split sizes — train: {len(train_ds):,}  val: {len(val_ds):,}  test: {len(test_ds):,}')

label_matrix = train_ds.label_matrix
counts = label_matrix.sum(axis=0)
total = len(label_matrix)

print('\nClass distribution (training set):')
print(f'{"Condition":<44} Count    %')
for name, c in zip(CONDITION_NAMES, counts):
    print(f'  {name:<42} {int(c):5d}  {100*c/total:.1f}%')

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#2ecc71' if c/total > 0.15 else '#f39c12' if c/total > 0.05 else '#e74c3c'
          for c in counts]
ax.barh([n.replace('Left ', 'L-').replace('Right ', 'R-') for n in CONDITION_NAMES],
        counts, color=colors)
ax.set_xlabel('Number of records')
ax.set_title('PTB-XL Training Set — Class Distribution')
for i, v in enumerate(counts):
    ax.text(v + 50, i, f'{v:.0f}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(f'{EVAL_DIR}/class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Inspect a sample ECG

In [ ]:
# ── Section 6: Inspect a sample ECG (reads wfdb directly — no resampling) ──
import wfdb, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv(Path(DATA_DIR) / 'ptbxl_database.csv')

# Auto-pick whichever resolution is actually present
rec = df.iloc[0]
sig, meta = None, None
for col in ['filename_hr', 'filename_lr']:
    path = str(Path(DATA_DIR) / rec[col])
    try:
        sig, meta = wfdb.rdsamp(path)
        print(f"Loaded via '{col}': shape={sig.shape}, fs={meta['fs']} Hz")
        break
    except Exception as e:
        print(f"'{col}' failed: {e}")

if sig is None:
    raise RuntimeError("Could not load any record — check DATA_DIR paths above.")

# Z-score each lead for display
sig = (sig - sig.mean(axis=0)) / (sig.std(axis=0) + 1e-8)
t = np.arange(sig.shape[0]) / meta['fs']

lead_names = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']
fig, axes = plt.subplots(6, 2, figsize=(14, 10), sharex=True)
axes = axes.flatten()
for i, ax in enumerate(axes):
    ax.plot(t, sig[:, i], linewidth=0.6, color='#2c3e50')
    ax.set_ylabel(lead_names[i], fontsize=9, rotation=0, labelpad=25)
    ax.set_ylim(-3, 3)
    ax.tick_params(labelsize=7)
    ax.axhline(0, color='#bdc3c7', linewidth=0.4)
    ax.grid(True, alpha=0.2)

axes[-1].set_xlabel('Time (s)')
fig.suptitle(f'Sample ECG — patient {rec["patient_id"]} | {meta["fs"]} Hz',
             fontsize=11)
plt.tight_layout()
plt.show()

## 7. Build model & verify architecture

In [ ]:
import torch
from ecg_model.model import build_model, ModelConfig, CONDITION_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

cfg = ModelConfig()  # defaults: 4 CNN blocks, 2 Transformer layers, 4 heads
model = build_model(cfg, device=str(device))

counts = model.parameter_count()
print(f'\nModel architecture:')
print(f'  Total parameters:     {counts["total"]:,}')
print(f'  Trainable parameters: {counts["trainable"]:,}')
print(f'  CNN channels:         {cfg.cnn_channels}')
print(f'  Transformer heads:    {cfg.n_heads}')
print(f'  Transformer layers:   {cfg.n_transformer_layers}')
print(f'  Tokens per forward:   {cfg.n_leads * model.t_prime}')

# Forward pass smoke test.
dummy = torch.randn(2, 12, 5000).to(device)
with torch.no_grad():
    logits = model(dummy)
    probs = model.predict_proba(dummy)
print(f'\nForward pass OK:')
print(f'  Input:   {tuple(dummy.shape)}')
print(f'  Logits:  {tuple(logits.shape)}')
print(f'  Probs range: [{probs.min().item():.3f}, {probs.max().item():.3f}]')
assert probs.shape == (2, len(CONDITION_NAMES))
assert probs.min().item() >= 0.0 and probs.max().item() <= 1.0
print('  Assertions passed ✓')

In [ ]:
# ── Patch dataset to handle 100 Hz records ────────────────────────────────
# If the diagnostic showed filename_lr / 100 Hz, the dataset.py is loading
# the wrong column. Override n_samples to match.
import wfdb, pandas as pd
from pathlib import Path

df = pd.read_csv(Path(DATA_DIR) / 'ptbxl_database.csv')
rec = df.iloc[0]

# Detect actual sample count
for col in ['filename_hr', 'filename_lr']:
    try:
        sig, meta = wfdb.rdsamp(str(Path(DATA_DIR) / rec[col]))
        ACTUAL_FS       = meta['fs']          # 100 or 500
        ACTUAL_SAMPLES  = sig.shape[0]        # 1000 or 5000
        ACTUAL_COL      = col
        print(f"Using '{col}': {ACTUAL_FS} Hz, {ACTUAL_SAMPLES} samples")
        break
    except:
        continue

## 8. Configure & run training

In [ ]:
from ecg_model.train import TrainConfig, train as run_training

from ecg_model.model import ModelConfig
cfg = TrainConfig(
    data_dir        = DATA_DIR,
    checkpoint_dir  = CHECKPOINT_DIR,
    log_dir         = LOG_DIR,
    epochs          = 50,
    batch_size      = 64,
    num_workers     = 2,
    lr              = 3e-4,
    weight_decay    = 1e-4,
    warmup_epochs   = 3,
    patience        = 7,
    pos_weight_cap  = 20.0,
    seed            = 42,
    model_cfg       = ModelConfig(n_samples=ACTUAL_SAMPLES),  # ← add this line
)

print('Training configuration:')
for k, v in vars(cfg).items():
    if k != 'model_cfg':
        print(f'  {k:<22}: {v}')
print()
print('Starting training... (checkpoints saved to Drive on every val-AUC improvement)')
print('The session can disconnect and resume — the best checkpoint persists in Drive.\n')

In [ ]:
import numpy as np
import ecg_model.dataset as D

def _time_warp(signal, rng, fs=500, max_warp=0.05):
    """Smooth time-warp augmentation — xp and fp are always the same length."""
    n_leads, n_samples = signal.shape
    xp = np.linspace(0, n_samples - 1, 6)
    shifts = rng.uniform(-max_warp * n_samples, max_warp * n_samples, 6)
    shift_curve = np.interp(np.arange(n_samples), xp, shifts)
    old_idx = np.arange(n_samples, dtype=np.float64)
    new_idx = np.clip(old_idx + shift_curve, 0, n_samples - 1)
    out = np.empty_like(signal, dtype=np.float64)
    for i in range(n_leads):
        out[i] = np.interp(new_idx, old_idx, signal[i])
    return out.astype(signal.dtype)

D._time_warp = _time_warp

# Smoke test
_sig = np.random.randn(12, ACTUAL_SAMPLES).astype(np.float32)
_out = D.augment_signal(_sig, fs=ACTUAL_FS)
assert _out.shape == _sig.shape, f"Shape mismatch: {_out.shape}"
print("✓ _time_warp fix applied and verified")


In [ ]:
import gc, torch, numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

gc.collect(); torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

from ecg_model.dataset import load_ptbxl
from ecg_model.model   import build_model, ModelConfig

BATCH_SIZE = 32
EPOCHS     = 50
PATIENCE   = 7

train_ds, val_ds, _ = load_ptbxl(cfg.data_dir)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

device  = torch.device('cuda')
model   = build_model(ModelConfig(n_samples=ACTUAL_SAMPLES), device='cuda')
optim   = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched   = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS, eta_min=1e-6)
scaler  = torch.amp.GradScaler('cuda')
crit    = nn.BCEWithLogitsLoss()

def compute_auc(probs, labels):
    aucs = [
        roc_auc_score(labels[:, c], probs[:, c])
        for c in range(labels.shape[1])
        if len(np.unique(labels[:, c])) >= 2
    ]
    return float(np.mean(aucs)) if aucs else float('nan')

best_auc, no_improve = 0.0, 0
print(f"\n{'Ep':>3}  {'Train-L':>8}  {'Val-L':>7}  {'Val-AUC':>8}  {'LR':>8}")
print("-" * 48)

for epoch in range(1, EPOCHS + 1):
    model.train()
    tloss, n = 0.0, 0
    bar = tqdm(train_loader, desc=f"Ep {epoch:02d}/{EPOCHS} train", leave=False,
               bar_format="{l_bar}{bar:20}{r_bar}")
    for sigs, labels in bar:
        sigs, labels = sigs.to(device), labels.to(device)
        optim.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            loss = crit(model(sigs), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optim); scaler.update()
        tloss += loss.item(); n += 1
        bar.set_postfix(loss=f"{loss.item():.3f}",
                        mem=f"{torch.cuda.memory_allocated()/1e9:.1f}G")
    sched.step()

    model.eval()
    vloss, probs_all, labels_all = 0.0, [], []
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for sigs, labels in val_loader:
            sigs, labels = sigs.to(device), labels.to(device)
            logits = model(sigs)
            vloss += crit(logits, labels).item()
            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            labels_all.append(labels.cpu().numpy())

    probs_all  = np.concatenate(probs_all)
    labels_all = np.concatenate(labels_all)
    auc = compute_auc(probs_all, labels_all)
    lr  = sched.get_last_lr()[0]
    print(f"{epoch:>3}  {tloss/n:>8.4f}  {vloss/len(val_loader):>7.4f}"
          f"  {auc:>8.4f}  {lr:>8.2e}")

    if auc > best_auc + 1e-4:
        best_auc, no_improve = auc, 0
        torch.save(model.state_dict(), f"{cfg.checkpoint_dir}/best_model.pt")
        print(f"           ↑ saved (AUC {auc:.4f})")
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)")
            break

print(f"\n✓ Training complete — best val AUC: {best_auc:.4f}")
print(f"  Checkpoint: {cfg.checkpoint_dir}/best_model.pt")


## 9. Plot training curves

In [ ]:
import json, matplotlib.pyplot as plt, numpy as np
from pathlib import Path
from ecg_model.evaluate import plot_training_history

history_path = f'{LOG_DIR}/history.json'
plot_training_history(history_path, EVAL_DIR)

with open(history_path) as f:
    history = json.load(f)

best = max(history, key=lambda h: h['val_auc'])
print(f'Best epoch:   {best["epoch"]}')
print(f'Best val AUC: {best["val_auc"]:.4f}')
print(f'Val loss:     {best["val_loss"]:.4f}')

# Show training curves inline.
from PIL import Image
img = Image.open(f'{EVAL_DIR}/training_curves.png')
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 10. Evaluate on held-out test set

In [ ]:
from ecg_model.evaluate import evaluate
from pathlib import Path

calibrated_path = f'{CHECKPOINT_DIR}/calibrated_model.pt'

summary = evaluate(
    checkpoint_path = calibrated_path,
    data_dir        = DATA_DIR,
    output_dir      = EVAL_DIR,
    batch_size      = 64,
    num_workers     = 2,
)

## 11. View calibration & ROC plots

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for fname, title in [
    ('roc_curves.png', 'ROC Curves'),
    ('calibration_curves.png', 'Calibration Curves (Reliability Diagrams)'),
    ('class_distribution.png', 'Class Distribution'),
]:
    path = f'{EVAL_DIR}/{fname}'
    try:
        img = Image.open(path)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(title)
        plt.tight_layout()
        plt.show()
    except FileNotFoundError:
        print(f'{fname} not found yet — run the evaluate cell first')

## 12. Quick inference test — single ECG

In [ ]:
import torch
from ecg_model.model import load_checkpoint, CONDITION_NAMES
from ecg_model.dataset import load_ptbxl

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = load_checkpoint(f'{CHECKPOINT_DIR}/calibrated_model.pt', device=str(device))
model.eval()

# Grab one record from the test set.
_, _, test_ds = load_ptbxl(DATA_DIR)
signal, true_labels = test_ds[0]

with torch.no_grad():
    probs = model.predict_proba(signal.unsqueeze(0).to(device))[0].cpu().numpy()

print(f'Single-record inference results:')
print(f'{"Condition":<44} Prob    True')
print('-' * 60)
for cond, prob, truth in zip(CONDITION_NAMES, probs, true_labels):
    bar = '█' * int(prob * 20) + '░' * (20 - int(prob * 20))
    marker = '✓' if truth == 1 else ' '
    print(f'  {cond:<42} [{bar}] {prob:.3f}  {marker}')

## 13. Copy final checkpoint for Phase 4 (API)
The API (Phase 4) will load `calibrated_model.pt` at startup. Copy it to a
stable location in Drive that the FastAPI Docker container can access.

In [ ]:
import shutil
from pathlib import Path

src = f'{CHECKPOINT_DIR}/calibrated_model.pt'
dst = f'{BASE_DIR}/model_release/calibrated_model.pt'
Path(dst).parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)
print(f'Checkpoint copied to: {dst}')
print(f'File size: {Path(dst).stat().st_size / 1e6:.1f} MB')
print()
print('Next steps:')
print('  1. Download calibrated_model.pt from Drive to your local machine')
print('  2. Place it in the phase4/api/model/ directory')
print('  3. Proceed to Phase 4 (FastAPI Cloud REST API)')

## Troubleshooting

**CUDA out of memory**
Reduce `batch_size` from 64 to 32 in the TrainConfig cell, then re-run.

**Session disconnected during training**
Re-run cells 1, 2, 3, 8 (the training cell). The best checkpoint is already
in Drive. Training will continue from epoch 1, but early stopping will
kick in once the new run can't beat the saved best.
For proper resume, add `torch.load(CHECKPOINT_DIR/best_model.pt)` and
restore optimizer state — a future improvement.

**PTB-XL download stalls**
Re-run cells 4a and 4b. The Kaggle download is resumable — already-extracted files in Drive are skipped automatically. Make sure your `kaggle.json` is valid and your Kaggle account has accepted the dataset terms at https://www.kaggle.com/datasets/khyeh0719/ptb-xl-dataset

**Low AUC on a specific condition**
Check the class distribution plot — very rare conditions (< 2%) need a
higher `pos_weight_cap` or focal loss. ST Elevation and PVC in particular
are underrepresented in PTB-XL.